In [ ]:
import numpy as np
import os
os.makedirs("figures", exist_ok=True)
from collections import defaultdict

means = np.array([1, 1.5, 1])
variances = np.array([1.5, 0.5, 1.5])
n_actions = 3 
pidata = np.array([ 0.04, 0.93, 0.03])
piref = np.array([0.00025, 0.00025, 0.9995])

beta = 0.001
    
def action_matrix(n):
    M = np.zeros((n, n), dtype=int)
    for i in range(n):
        M[i, i] = 1               # diagonal
        M[i, (i+1)%n] = -1        # next column (wrap around)
    return M
def balanced_chunks(rng, first_actions, second_actions, winner_indices, L, seed=None):
    """
    Split three aligned vectors (first_actions, second_actions, winner_indices)
    into L chunks such that:
    - Unordered pairs (min(a,b), max(a,b)) are distributed as evenly as possible.
    - The difference in counts of any pair across chunks <= 1
    - Chunks are reproducible with a seed.
    
    Returns:
        first_chunks, second_chunks, winner_chunks : lists of length L
    """
    first_actions = np.asarray(first_actions)
    second_actions = np.asarray(second_actions)
    winner_indices = np.asarray(winner_indices)
    
    if not (len(first_actions) == len(second_actions) == len(winner_indices)):
        raise ValueError("All input vectors must have the same length")
    
    
    
    # Step 1: represent unordered pairs
    pairs = np.stack([np.minimum(first_actions, second_actions),
                      np.maximum(first_actions, second_actions)], axis=1)
    
    # Step 2: map pairs to their indices
    pair_to_indices = defaultdict(list)
    for idx, (a, b) in enumerate(pairs):
        pair_to_indices[(a, b)].append(idx)
    
    # Step 3: prepare empty chunks
    chunks_indices = [[] for _ in range(L)]
    
    # Step 4: distribute indices of each pair in round-robin
    for idx_list in pair_to_indices.values():
        idx_list = rng.permutation(idx_list)  # shuffle for randomness
        for i, idx in enumerate(idx_list):
            chunk_id = i % L
            chunks_indices[chunk_id].append(idx)
    
    # Step 5: sort indices within chunks (optional) to preserve order
    # chunks_indices = [sorted(chunk) for chunk in chunks_indices]
    
    # Step 6: extract chunks for each vector
    first_chunks = [first_actions[chunk] for chunk in chunks_indices]
    second_chunks = [second_actions[chunk] for chunk in chunks_indices]
    winner_chunks = [winner_indices[chunk] for chunk in chunks_indices]
    
    return first_chunks, second_chunks, winner_chunks
def sigmoid(x):
    return np.where(
        x >= 0,
        1 / (1 + np.exp(-x)),
        np.exp(x) / (1 + np.exp(x))
    )

def softmax(x, axis=-1):
    x = np.asarray(x, dtype=float)

    # subtract max for numerical stability
    x_shifted = x - np.max(x, axis=axis, keepdims=True)

    exp_x = np.exp(x_shifted)
    return exp_x / np.sum(exp_x, axis=axis, keepdims=True)

def weird_softmax(x, gamma=30000, axis=-1):
    x = np.asarray(x, dtype=float)
    
    # shift max for stability (like normal softmax)
    x_shifted = x - np.max(x, axis=axis, keepdims=True)
    
    vals = inv_gamma_x_plus_logx(x_shifted, gamma)
    return vals / np.sum(vals, axis=axis, keepdims=True)

def inv_gamma_x_plus_logx(y, gamma, max_iter=20):
    y = np.asarray(y, dtype=float)
    
    # initial guess: large y → y/gamma, small y → exp(y)
    x = np.where(y > 1, y/gamma, np.exp(y))
    
    for _ in range(max_iter):
        f = gamma*x + np.log(x) - y      # f(x) = gamma*x + log(x) - y
        fp = gamma + 1/x                  # f'(x) = gamma + 1/x
        x -= f / fp
    
    return x

def create_dataset(rng, pidata, means, variances, N):
    first_actions = np.random.choice(len(means), size=N, replace=True,p=pidata)
    second_actions = np.random.choice(len(means), size=N, replace=True,p=pidata)
    first_rewards = [rng.normal(means[a],variances[a]) for a in first_actions]
    second_rewards = [rng.normal(means[a],variances[a]) for a in second_actions]    
    bernoulli_means = np.array(first_rewards) - np.array(second_rewards)
    winner_indices = rng.binomial(n=1, p=sigmoid(bernoulli_means))
    return first_actions, second_actions, winner_indices

def compute_performances(policies, means, beta, piref):
    return [ policy.dot(means - beta*np.log(policy/piref)) for policy in policies]
def compute_subopts(policies, means, beta, piref):
    return [ np.max(means) - policy.dot(means)  for policy in policies]    

def compute_win_rates(first_actions, second_actions, winner_indices, bias=2):
    output = []

    for first, second in [(0, 1), (1, 2), (2, 0)]:
        # elementwise logical AND, not Python "and"
        mask = (first_actions == first) & (second_actions == second)

        total = np.sum(mask)                 # how many such matchups
        wins = np.sum(winner_indices[mask])  # assumes winner_indices is 0/1
        
        mask2 = (first_actions == second) & (second_actions == first)
        total += np.sum(mask2) 
        wins += (np.sum(mask2) - np.sum(winner_indices[mask2]) )
        rate = wins / (total + bias)
        output.append(rate)

    return output

def DPO(first_actions, second_actions, winner_indices,piref):
    target = compute_win_rates(first_actions, second_actions, winner_indices,bias=0.01)
    A = action_matrix(n_actions)
    logratios, *_ = np.linalg.lstsq(A, target, rcond=None)
    policy = softmax(logratios/beta + np.log(piref))
    return policy

def chi2PO(first_actions, second_actions, winner_indices,piref,gamma=40):
    target = compute_win_rates(first_actions, second_actions, winner_indices,bias=0.01)
    A = action_matrix(n_actions)
    logratios, *_ = np.linalg.lstsq(A, target, rcond=None)
    policy = weird_softmax(logratios/beta + np.log(piref) + gamma*piref, gamma)
    return policy

def PEPO(rng, first_actions, second_actions, winner_indices,piref, L=1):
    
    f_chunks, s_chunks, w_chunks = balanced_chunks(rng, first_actions, second_actions, winner_indices, L)
    
    policies = []
    for c_1, c_2, c_3 in zip(f_chunks, s_chunks, w_chunks):
        target = compute_win_rates(c_1, c_2, c_3,bias=0.01)
        A = action_matrix(n_actions)
        logratios, *_ = np.linalg.lstsq(A, target, rcond=None)
        policies.append(softmax(logratios/beta + np.log(piref)))
    
    num_out = np.min(policies, axis=0)
    
    policy = num_out/num_out.sum()
    return policy 
import numpy as np

def dpo_sft_gradient_update(logits, first_actions, second_actions, winner_indices, ref_probs, lr=0.01, beta=0.1, alpha=0.5):
    """
    logits: 1D array of current logits for each action [num_actions]
    first_actions: 1D array of indices for the first action in each pair [batch_size]
    second_actions: 1D array of indices for the second action in each pair [batch_size]
    winner_indices: 1D array of 1s (first wins) or 0s (second wins) [batch_size]
    ref_probs: 1D array of reference probabilities for each action [num_actions]
    """
    batch_size = len(winner_indices)
    num_actions = len(logits)
    
    # 1. Map winners and losers for the whole batch
    # If winner_indices == 1, w = first, l = second. Else w = second, l = first.
    w_idxs = np.where(winner_indices == 1, first_actions, second_actions)
    l_idxs = np.where(winner_indices == 1, second_actions, first_actions)

    # 2. Get current probabilities (Softmax)
    probs = softmax(logits)
    
    # Extract specific probs for the winners/losers in this batch
    pi_w, pi_l = probs[w_idxs], probs[l_idxs]
    ref_w, ref_l = ref_probs[w_idxs], ref_probs[l_idxs]

    # 3. Compute DPO weights for the batch
    # diff_log_ratio = beta * [ log(pi_w/ref_w) - log(pi_l/ref_l) ]
    diff_log_ratio = beta * ((np.log(pi_w) - np.log(ref_w)) - (np.log(pi_l) - np.log(ref_l)))
    # Weight per sample: -beta * sigmoid(-beta * diff)
    weights = -beta * (1.0 / (1.0 + np.exp(diff_log_ratio)))

    # 4. Vectorized Gradient Accumulation
    grad = np.zeros_like(logits)

    # --- SFT Component Gradient ---
    # The SFT loss is -log(pi_w). 
    # Average Gradient for SFT across batch: (1/N) * sum(probs - target)
    # target is a one-hot vector for the winner.
    for i in range(batch_size):
        # Contribution from SFT
        grad += alpha * probs / batch_size
        grad[w_idxs[i]] -= alpha / batch_size
        
        # Contribution from DPO
        # DPO gradient for action 'a': weight * (I(a=w) - I(a=l)) * pi_a (simplified for tabular)
        # More accurately for the winner/loser logits:
        grad[w_idxs[i]] += (weights[i] * (1 - pi_w[i])) / batch_size
        grad[l_idxs[i]] -= (weights[i] * pi_l[i]) / batch_size

    # 5. Update
    new_logits = logits - lr * grad
    return new_logits

def DPO_SFT(first_actions, second_actions, winner_indices, piref):
    
    logits = piref
    for epoch in range(20):
        #w_idx = [ first_actions[i] if win_bit == 1 else second_actions[i] for i, win_bit in enumerate(winner_indices)]
        #l_idx = [ first_actions[i] if win_bit == 0 else second_actions[i] for i, win_bit in enumerate(winner_indices)]
        # Update the logits using the function from the previous step
        logits = dpo_sft_gradient_update(logits, first_actions, second_actions, winner_indices, piref)
    return softmax(logits)

optimal_policy = softmax(means/beta + np.log(piref))

In [ ]:
def run(N = 10000,seed=0):
    
    rng = np.random.default_rng(seed)
    np.random.seed(seed)
    
    first_actions, second_actions, winner_indices = create_dataset(rng, pidata, means, variances, N)
    
    DPOpolicy = DPO(first_actions, second_actions, winner_indices,piref)
    chi2POpolicy = chi2PO(first_actions, second_actions, winner_indices,piref)
    PEPOpolicy = PEPO(rng, first_actions, second_actions, winner_indices, piref,L=5)
    #DPOSFTpolicy = DPO_SFT(first_actions, second_actions, winner_indices, piref)
    return DPOpolicy, chi2POpolicy, PEPOpolicy #, DPOSFTpolicy

In [ ]:
unif_perf = compute_performances([np.ones(n_actions)/n_actions], means, beta, piref)

perfs = {"DPOmeans": [unif_perf[0]], "chi2POmeans": [unif_perf[0]], "PEPOmeans": [unif_perf[0]], #"DPOSFTmeans": [unif_perf[0]],
        "DPOstds": [0], "chi2POstds": [0], "PEPOstds": [0] #, "DPOSFTstds": [0]
        }
subopts = {"DPOmeans": [unif_perf[0]], "chi2POmeans": [unif_perf[0]], "PEPOmeans": [unif_perf[0]], #"DPOSFTmeans": [unif_perf[0]],
        "DPOstds": [0], "chi2POstds": [0], "PEPOstds": [0], #"DPOSFTstds": [0]
          }
N_values = [150, 500, 1000, 1500, 2000, 3000, 4000, 5000, 6000]
for N in N_values:
    DPOperfs = []
    chi2POperfs = []
    PEPOperfs = []
    #DPOSFTperfs = []
    DPOsubopts = []
    chi2POsubopts = []
    PEPOsubopts = []
    #DPOSFTsubopts = []
    for seed in range(10):
        policies = run(seed=seed,N=N)
        DPOperf, chi2POperf, PEPOperf = compute_performances(policies, means, beta, piref)
        DPOsubopt, chi2POsubopt, PEPOsubopt = compute_subopts(policies, means, beta, piref)
        DPOperfs.append(DPOperf)
        chi2POperfs.append(chi2POperf)
        PEPOperfs.append(PEPOperf)
        #DPOSFTperfs.append(DPOSFTperf)
        DPOsubopts.append(DPOsubopt)
        chi2POsubopts.append(chi2POsubopt)
        PEPOsubopts.append(PEPOsubopt)
        #DPOSFTsubopts.append(DPOSFTsubopt)
    perfs["DPOmeans"].append(np.mean(DPOperfs))
    perfs["chi2POmeans"].append(np.mean(chi2POperfs))
    perfs["PEPOmeans"].append(np.mean(PEPOperfs))
    #perfs["DPOSFTmeans"].append(np.mean(DPOSFTperfs))
    perfs["DPOstds"].append(np.std(DPOperfs))    
    perfs["chi2POstds"].append(np.std(chi2POperfs))     
    perfs["PEPOstds"].append(np.std(PEPOperfs)) 
    #perfs["DPOSFTstds"].append(np.mean(DPOSFTperfs))
    subopts["DPOmeans"].append(np.mean(DPOsubopts))
    subopts["chi2POmeans"].append(np.mean(chi2POsubopts))
    subopts["PEPOmeans"].append(np.mean(PEPOsubopts))
    #subopts["DPOSFTmeans"].append(np.mean(DPOSFTsubopts))
    subopts["DPOstds"].append(np.std(DPOsubopts))    
    subopts["chi2POstds"].append(np.std(chi2POsubopts))     
    subopts["PEPOstds"].append(np.std(PEPOsubopts))     
    #subopts["DPOSFTstds"].append(np.std(DPOSFTsubopts)) 

In [ ]:
import matplotlib.pyplot as plt

plt.rcParams.update({
    "text.usetex": True,      # enable LaTeX rendering
    "font.family": "serif",   # choose a LaTeX-like font
    "axes.labelsize": 20,
    "xtick.labelsize": 20,
    "ytick.labelsize": 20,
    "legend.fontsize": 20,
})
for data,name in zip([perfs], ["perfs"]):
    plt.figure()
    plt.grid()
    optimal_perf = compute_performances([optimal_policy], means, beta, piref)
    N_values_new = np.concatenate(([0], N_values))

    # Plot with explicit x-values
    plt.plot(N_values_new, optimal_perf[0] - np.array(data["DPOmeans"]), label="DPO", color="orange")
    plt.plot(N_values_new, optimal_perf[0] - np.array(data["PEPOmeans"]), label="PEPO", color="red")
    plt.plot(N_values_new, optimal_perf[0] - np.array(data["chi2POmeans"]), label=r"$\chi^2$PO", color="blue")
    plt.fill_between(N_values_new, optimal_perf[0] - np.array(data["DPOmeans"]) - 0.1*np.array(data["DPOstds"]),
     optimal_perf[0] - np.array(data["DPOmeans"]) + 0.1*np.array(data["DPOstds"]), alpha=0.1, color="orange")
    plt.fill_between(N_values_new, optimal_perf[0] - np.array(data["chi2POmeans"]) - 0.1*np.array(data["chi2POstds"]),
     optimal_perf[0] - np.array(data["chi2POmeans"]) + 0.1*np.array(data["chi2POstds"]), alpha=0.1, color="blue")
    plt.fill_between(N_values_new, optimal_perf[0] - np.array(data["PEPOmeans"]) - 0.1*np.array(data["PEPOstds"]),
     optimal_perf[0] - np.array(data["PEPOmeans"]) + 0.1*np.array(data["PEPOstds"]), alpha=0.1, color="red")
    # Set xticks at desired positions
    #plt.xticks(N_values)
    ylabel = r"$J_\beta(\pi^\star) - J_\beta(\pi_{\mathrm{out}})$" if name == "perfs" else r"$\langle \pi^\star - \pi_{\mathrm{out}}, r^\star \rangle$"
    
    plt.ylabel(ylabel)
    plt.xlabel("Dataset size")
    plt.legend()
    plt.tight_layout()
    plt.savefig(f"figures/{name}pidataneqpiref.pdf")
    plt.show()

In [ ]:
for data,name in zip([subopts], ["subopts"]):
    plt.figure()
    plt.grid()
    N_values_new = np.concatenate(([0], N_values))

    # Plot with explicit x-values
    plt.plot(N_values_new, np.array(data["DPOmeans"]), label="DPO", color="orange")
    plt.plot(N_values_new, np.array(data["PEPOmeans"]), label="PEPO", color="red")
    plt.plot(N_values_new, np.array(data["chi2POmeans"]), label=r"$\chi^2$PO", color="blue")
    plt.fill_between(N_values_new, np.array(data["DPOmeans"]) - 0.1*np.array(data["DPOstds"]),
     np.array(data["DPOmeans"]) + 0.1*np.array(data["DPOstds"]), alpha=0.1, color="orange")
    plt.fill_between(N_values_new,  np.array(data["chi2POmeans"]) - 0.1*np.array(data["chi2POstds"]),
     np.array(data["chi2POmeans"]) + 0.1*np.array(data["chi2POstds"]), alpha=0.1, color="blue")
    plt.fill_between(N_values_new, np.array(data["PEPOmeans"]) - 0.1*np.array(data["PEPOstds"]),
     np.array(data["PEPOmeans"]) + 0.1*np.array(data["PEPOstds"]), alpha=0.1, color="red")
    # Set xticks at desired positions
    #plt.xticks(N_values)
    ylabel = r"$J_\beta(\pi^\star) - J_\beta(\pi_{\mathrm{out}})$" if name == "perfs" else r"$\langle \pi^\star - \pi_{\mathrm{out}}, r^\star \rangle$"
    
    plt.ylabel(ylabel)
    plt.xlabel("Dataset size")
    plt.legend()
    plt.tight_layout()
    plt.savefig(f"figures/{name}pidataneqpiref.pdf")
    plt.show()

In [ ]:
means = np.array([1, 1.5, 1])
variances = np.array([1.5, 0.5, 1.5])
n_actions = 3 
pidata = np.array([ 0.04, 0.93, 0.03])
piref = pidata 

beta = 0.001

optimal_policy = softmax(means/beta + np.log(piref))

In [ ]:
def run(N = 10000,seed=0):
    
    rng = np.random.default_rng(seed)
    np.random.seed(seed)
    
    first_actions, second_actions, winner_indices = create_dataset(rng, pidata, means, variances, N)
    
    DPOpolicy = DPO(first_actions, second_actions, winner_indices,piref)
    chi2POpolicy = chi2PO(first_actions, second_actions, winner_indices,piref)
    PEPOpolicy = PEPO(rng, first_actions, second_actions, winner_indices,piref,L=5)
    return DPOpolicy, chi2POpolicy, PEPOpolicy

In [ ]:
unif_perf = compute_performances([np.ones(n_actions)/n_actions], means, beta, piref)

perfs = {"DPOmeans": [unif_perf[0]], "chi2POmeans": [unif_perf[0]], "PEPOmeans": [unif_perf[0]],
        "DPOstds": [0], "chi2POstds": [0], "PEPOstds": [0]}
subopts = {"DPOmeans": [unif_perf[0]], "chi2POmeans": [unif_perf[0]], "PEPOmeans": [unif_perf[0]],
        "DPOstds": [0], "chi2POstds": [0], "PEPOstds": [0]}
N_values = [150, 500, 1000, 1500, 2000, 3000, 4000, 5000, 6000]
for N in N_values:
    DPOperfs = []
    chi2POperfs = []
    PEPOperfs = []
    DPOsubopts = []
    chi2POsubopts = []
    PEPOsubopts = []
    for seed in range(10):
        policies = run(seed=seed,N=N)
        DPOperf, chi2POperf, PEPOperf = compute_performances(policies, means, beta, piref)
        DPOsubopt, chi2POsubopt, PEPOsubopt = compute_subopts(policies, means, beta, piref)
        DPOperfs.append(DPOperf)
        chi2POperfs.append(chi2POperf)
        PEPOperfs.append(PEPOperf)
        DPOsubopts.append(DPOsubopt)
        chi2POsubopts.append(chi2POsubopt)
        PEPOsubopts.append(PEPOsubopt)
    perfs["DPOmeans"].append(np.mean(DPOperfs))
    perfs["chi2POmeans"].append(np.mean(chi2POperfs))
    perfs["PEPOmeans"].append(np.mean(PEPOperfs))
    perfs["DPOstds"].append(np.std(DPOperfs))    
    perfs["chi2POstds"].append(np.std(chi2POperfs))     
    perfs["PEPOstds"].append(np.std(PEPOperfs)) 
    subopts["DPOmeans"].append(np.mean(DPOsubopts))
    subopts["chi2POmeans"].append(np.mean(chi2POsubopts))
    subopts["PEPOmeans"].append(np.mean(PEPOsubopts))
    subopts["DPOstds"].append(np.std(DPOsubopts))    
    subopts["chi2POstds"].append(np.std(chi2POsubopts))     
    subopts["PEPOstds"].append(np.std(PEPOsubopts)) 

In [ ]:
import matplotlib.pyplot as plt

plt.rcParams.update({
    "text.usetex": True,      # enable LaTeX rendering
    "font.family": "serif",   # choose a LaTeX-like font
    "axes.labelsize": 20,
    "xtick.labelsize": 20,
    "ytick.labelsize": 20,
    "legend.fontsize": 20,
})
for data,name in zip([perfs], ["perfs"]):
    plt.figure()
    plt.grid()
    optimal_perf = compute_performances([optimal_policy], means, beta, piref)
    N_values_new = np.concatenate(([0], N_values))

    # Plot with explicit x-values
    plt.plot(N_values_new, optimal_perf[0] - np.array(data["DPOmeans"]), label="DPO", color="orange")
    plt.plot(N_values_new, optimal_perf[0] - np.array(data["PEPOmeans"]), label="PEPO", color="red")
    plt.plot(N_values_new, optimal_perf[0] - np.array(data["chi2POmeans"]), label=r"$\chi^2$PO", color="blue")
    plt.fill_between(N_values_new, optimal_perf[0] - np.array(data["DPOmeans"]) - 0.1*np.array(data["DPOstds"]),
     optimal_perf[0] - np.array(data["DPOmeans"]) + 0.1*np.array(data["DPOstds"]), alpha=0.1, color="orange")
    plt.fill_between(N_values_new, optimal_perf[0] - np.array(data["chi2POmeans"]) - 0.1*np.array(data["chi2POstds"]),
     optimal_perf[0] - np.array(data["chi2POmeans"]) + 0.1*np.array(data["chi2POstds"]), alpha=0.1, color="blue")
    plt.fill_between(N_values_new, optimal_perf[0] - np.array(data["PEPOmeans"]) - 0.1*np.array(data["PEPOstds"]),
     optimal_perf[0] - np.array(data["PEPOmeans"]) + 0.1*np.array(data["PEPOstds"]), alpha=0.1, color="red")
    # Set xticks at desired positions
    
    ylabel = r"$J_\beta(\pi^\star) - J_\beta(\pi_{\mathrm{out}})$" if name == "perfs" else r"$\langle \pi^\star - \pi_{\mathrm{out}}, r^\star \rangle$"
    
    plt.ylabel(ylabel)
    plt.xlabel("Dataset size")
    #plt.legend()
    plt.tight_layout()
    plt.savefig(f"figures/{name}pidataeqpiref.pdf")
    plt.show()

In [ ]:
for data,name in zip([subopts], ["subopts"]):
    plt.figure()
    plt.grid()
    N_values_new = np.concatenate(([0], N_values))

    # Plot with explicit x-values
    plt.plot(N_values_new, np.array(data["DPOmeans"]), label="DPO", color="orange")
    plt.plot(N_values_new, np.array(data["PEPOmeans"]), label="PEPO", color="red")
    plt.plot(N_values_new, np.array(data["chi2POmeans"]), label=r"$\chi^2$PO", color="blue")
    plt.fill_between(N_values_new, np.array(data["DPOmeans"]) - 0.1*np.array(data["DPOstds"]),
     np.array(data["DPOmeans"]) + 0.1*np.array(data["DPOstds"]), alpha=0.1, color="orange")
    plt.fill_between(N_values_new,  np.array(data["chi2POmeans"]) - 0.1*np.array(data["chi2POstds"]),
     np.array(data["chi2POmeans"]) + 0.1*np.array(data["chi2POstds"]), alpha=0.1, color="blue")
    plt.fill_between(N_values_new, np.array(data["PEPOmeans"]) - 0.1*np.array(data["PEPOstds"]),
     np.array(data["PEPOmeans"]) + 0.1*np.array(data["PEPOstds"]), alpha=0.1, color="red")
    # Set xticks at desired positions
    
    ylabel = r"$J_\beta(\pi^\star) - J_\beta(\pi_{\mathrm{out}})$" if name == "perfs" else r"$\langle \pi^\star - \pi_{\mathrm{out}}, r^\star \rangle$"
    
    plt.ylabel(ylabel)
    plt.xlabel("Dataset size")
    plt.legend()
    plt.tight_layout()
    plt.savefig(f"figures/{name}pidataeqpiref.pdf")
    plt.show()

# Greedy experiment

In [ ]:
def relu(x):
    return np.maximum(0, x)

def DPOgreedy(first_actions, second_actions, winner_indices,piref):
    target = compute_win_rates(first_actions, second_actions, winner_indices,bias=0.01)
    A = action_matrix(n_actions)
    logratios, *_ = np.linalg.lstsq(A, target, rcond=None)
    policy = np.zeros_like(logratios)
    policy[np.argmax(logratios)] = 1
    return policy

def chi2POgreedy(first_actions, second_actions, winner_indices,piref,gamma=40):
    target = compute_win_rates(first_actions, second_actions, winner_indices,bias=0.01)
    A = action_matrix(n_actions)
    logratios, *_ = np.linalg.lstsq(A, target, rcond=None)
    return weird_softmax(logratios/0.001 + (np.log(piref) + gamma*piref) ,gamma)
    

def PEPOgreedy(rng, first_actions, second_actions, winner_indices,piref, L=1):
    
    f_chunks, s_chunks, w_chunks = balanced_chunks(rng, first_actions, second_actions, winner_indices, L)
    
    logratios_list = []
    for c_1, c_2, c_3 in zip(f_chunks, s_chunks, w_chunks):
        target = compute_win_rates(c_1, c_2, c_3,bias=0.01)
        A = action_matrix(n_actions)
        logratios, *_ = np.linalg.lstsq(A, target, rcond=None)
        logratios_list.append(logratios)
    
    logratio_min = np.min(logratios_list, axis=0)
    
    policy = np.zeros_like(logratio_min)
    policy[np.argmax(logratio_min)] = 1
    return policy 

In [ ]:
def run_greedy(N = 10000,seed=0):
    
    rng = np.random.default_rng(seed)
    np.random.seed(seed)
    
    first_actions, second_actions, winner_indices = create_dataset(rng, pidata, means, variances, N)
    
    DPOpolicy = DPOgreedy(first_actions, second_actions, winner_indices,piref)
    chi2POpolicy = chi2POgreedy(first_actions, second_actions, winner_indices,piref)
    PEPOpolicy = PEPOgreedy(rng, first_actions, second_actions, winner_indices,piref,L=3)
    return DPOpolicy, chi2POpolicy, PEPOpolicy

In [ ]:


#means = np.array([1, 1.5, 1])
#variances = np.array([1.5, 0.5, 1.5])
#n_actions = 3 
#pidata = np.array([0.1, 0.1, 0.8]) # [ 0.7, 0.25, 0.05]
#piref = pidata

means = np.array([1.4, 1.5, 1.4])
variances = np.array([1.5, 1, 1.5])
n_actions = 3 
pidata = np.array([ 0.14, 0.73, 0.13]) #pidata = np.array([ 0.43, 0.14, 0.43])
piref = pidata

unif_subopt = compute_subopts([np.ones(n_actions)/n_actions], means, beta, piref)

subopts = {"DPOmeans": [unif_subopt[0]], "chi2POmeans": [unif_subopt[0]], "PEPOmeans": [unif_subopt[0]],
        "DPOstds": [0], "chi2POstds": [0], "PEPOstds": [0]}
N_values = [150, 500, 1000, 1500, 2000, 3000, 4000, 5000, 6000, 7000, 8000]

for N in N_values:
    
    DPOsubopts = []
    chi2POsubopts = []
    PEPOsubopts = []
    for seed in range(10):
        policies = run_greedy(seed=seed,N=N)
        DPOsubopt, chi2POsubopt, PEPOsubopt = compute_subopts(policies, means, beta, piref)
        
        DPOsubopts.append(DPOsubopt)
        chi2POsubopts.append(chi2POsubopt)
        PEPOsubopts.append(PEPOsubopt)
    subopts["DPOmeans"].append(np.mean(DPOsubopts))
    subopts["chi2POmeans"].append(np.mean(chi2POsubopts))
    subopts["PEPOmeans"].append(np.mean(PEPOsubopts))
    subopts["DPOstds"].append(np.std(DPOsubopts))    
    subopts["chi2POstds"].append(np.std(chi2POsubopts))     
    subopts["PEPOstds"].append(np.std(PEPOsubopts)) 

In [ ]:
import matplotlib.pyplot as plt

plt.rcParams.update({
    "text.usetex": True,      # enable LaTeX rendering
    "font.family": "serif",   # choose a LaTeX-like font
    "axes.labelsize": 20,
    "xtick.labelsize": 20,
    "ytick.labelsize": 20,
    "legend.fontsize": 20,
})
for data,name in zip([subopts], ["subopts"]):
    plt.figure()
    plt.grid()
    N_values_new = np.concatenate(([0], N_values))

    # Plot with explicit x-values
    plt.plot(N_values_new, np.array(data["DPOmeans"]), label="RL", color="orange")
    plt.plot(N_values_new, np.array(data["PEPOmeans"]), label="PERL", color="red")
    plt.plot(N_values_new, np.array(data["chi2POmeans"]), label=r"$\chi^2$RL", color="blue")
    plt.fill_between(N_values_new, np.array(data["DPOmeans"]) - 0.1*np.array(data["DPOstds"]),
     np.array(data["DPOmeans"]) + 0.1*np.array(data["DPOstds"]), alpha=0.1, color="orange")
    plt.fill_between(N_values_new, np.array(data["chi2POmeans"]) - 0.1*np.array(data["chi2POstds"]),
     np.array(data["chi2POmeans"]) + 0.1*np.array(data["chi2POstds"]), alpha=0.1, color="blue")
    plt.fill_between(N_values_new, np.array(data["PEPOmeans"]) - 0.1*np.array(data["PEPOstds"]),
     np.array(data["PEPOmeans"]) + 0.1*np.array(data["PEPOstds"]), alpha=0.1, color="red")
    # Set xticks at desired positions
    
    ylabel = r"$J_\beta(\pi^\star) - J_\beta(\pi_{\mathrm{out}})$" if name == "perfs" else r"$\langle \pi^\star - \pi_{\mathrm{out}}, r^\star \rangle$"
    plt.ylabel(ylabel)
    plt.xlabel("Dataset size")
    plt.legend()
    plt.tight_layout()
    plt.savefig(f"figures/greedypidataeqpiref.pdf")
    plt.show()

In [ ]:
means = np.array([1.4, 1.5, 1.4])
variances = np.array([1.5, 1, 1.5])
n_actions = 3 
pidata = np.array([ 0.14, 0.73, 0.13])
piref = np.array([0.1, 0.1, 0.8])

unif_subopt = compute_subopts([np.ones(n_actions)/n_actions], means, beta, piref)

subopts = {"DPOmeans": [unif_subopt[0]], "chi2POmeans": [unif_subopt[0]], "PEPOmeans": [unif_subopt[0]],
        "DPOstds": [0], "chi2POstds": [0], "PEPOstds": [0]}
N_values = [150, 500, 1000, 1500, 2000, 3000, 4000, 5000, 6000, 7000, 8000]

for N in N_values:
    DPOsubopts = []
    chi2POsubopts = []
    PEPOsubopts = []
    for seed in range(10):
        policies = run_greedy(seed=seed,N=N)
        DPOsubopt, chi2POsubopt, PEPOsubopt = compute_subopts(policies, means, beta, piref)
        
        DPOsubopts.append(DPOsubopt)
        chi2POsubopts.append(chi2POsubopt)
        PEPOsubopts.append(PEPOsubopt)
    subopts["DPOmeans"].append(np.mean(DPOsubopts))
    subopts["chi2POmeans"].append(np.mean(chi2POsubopts))
    subopts["PEPOmeans"].append(np.mean(PEPOsubopts))
    subopts["DPOstds"].append(np.std(DPOsubopts))    
    subopts["chi2POstds"].append(np.std(chi2POsubopts))     
    subopts["PEPOstds"].append(np.std(PEPOsubopts)) 

In [ ]:
import matplotlib.pyplot as plt

plt.rcParams.update({
    "text.usetex": True,      # enable LaTeX rendering
    "font.family": "serif",   # choose a LaTeX-like font
    "axes.labelsize": 20,
    "xtick.labelsize": 20,
    "ytick.labelsize": 20,
    "legend.fontsize": 20,
})
for data,name in zip([subopts], ["subopts"]):
    plt.figure()
    plt.grid()
    N_values_new = np.concatenate(([0], N_values))

    # Plot with explicit x-values
    plt.plot(N_values_new, np.array(data["DPOmeans"]), label="DPO", color="orange")
    plt.plot(N_values_new, np.array(data["PEPOmeans"]), label="PEPO", color="red")
    plt.plot(N_values_new, np.array(data["chi2POmeans"]), label=r"$\chi^2$PO", color="blue")
    plt.fill_between(N_values_new, np.array(data["DPOmeans"]) - 0.1*np.array(data["DPOstds"]),
     np.array(data["DPOmeans"]) + 0.1*np.array(data["DPOstds"]), alpha=0.1, color="orange")
    plt.fill_between(N_values_new, np.array(data["chi2POmeans"]) - 0.1*np.array(data["chi2POstds"]),
     np.array(data["chi2POmeans"]) + 0.1*np.array(data["chi2POstds"]), alpha=0.1, color="blue")
    plt.fill_between(N_values_new, np.array(data["PEPOmeans"]) - 0.1*np.array(data["PEPOstds"]),
     np.array(data["PEPOmeans"]) + 0.1*np.array(data["PEPOstds"]), alpha=0.1, color="red")
    # Set xticks at desired positions
    ylabel = r"$J_\beta(\pi^\star) - J_\beta(\pi_{\mathrm{out}})$" if name == "perfs" else r"$\langle \pi^\star - \pi_{\mathrm{out}}, r^\star \rangle$"
    plt.ylabel(ylabel)
    plt.xlabel("Dataset size")
    #plt.legend()
    plt.tight_layout()
    plt.savefig(f"figures/greedypidataneqpiref.pdf")
    plt.show()